在本代码文件中，我们将从零实现一个GPT模型。

在开始之前，我们需要了解即将实现的GPT2整体架构:

![GPT整体架构图](Images/GPT-2%20Overall%20Architecture.svg)

图注：上图为GPT-2的整体架构图，GPT-2是Only-Decoder(仅解码器)架构的自回归模型，依靠堆叠多层Transformer Block捕捉上下文依赖的关系，完成从上文预测下一个Token的任务，整体流程分为输入编码、多层Transformer Block特征提取和输出预测三个阶段：
1. **输入编码**

原始输入文本先经过BPE分词器，拆分成子词Token，接着分词结果分别送入WTE(Word Token Embedding)词嵌入和WPE(Word Position Embedding)位置嵌入，WTE把离散的词嵌入映射为连续的语义向量，WPE则为语义向量注入位置信息，之后将词嵌入向量和位置嵌入向量直接相加，再经过Dropout做正则化，得到输入序列融合了语义信息和位置信息的初始特征向量，送入后续的Transformer模块做特征提取。

2. **多层Transformer Block特征提取**

GPT-2的标准版包含12个完全相同、串行堆叠的Transformer Block，每一个Transformer Block都采用Pre-LN(前置层归一化)设计，包含因果多头自注意力模块和MLP前馈网络两个模块，并且都使用残差连接。原始输入x先经过LayerNorm进行层归一化，再送入因果多头自注意力模块，通过掩码保证只能看到当前和先前的Token，实现子回归，捕捉上下文的依赖关系，经过Dropout后与原始输入x直接相加(残差连接)，得到的结果作为原始MLP输入，再次经过LayerNorm进行层归一化后，送入前馈神经网络(两个全连接层和GELU激活函数)，对单个Token的特征做非线性变换，再次经过Dropout后，与原始MLP输入直接相加，得到该Transformer Block的输出，送入下一个Block。

3. **输出预测**

经过12个Transformer Block之后，再做一次LayerNorm,随后进入LM Head线性投影层，将隐状态向量映射到词表维度，得到每个候选Token的logits,经过Softmax转换为概率分布，根据采样策略选择下一个Token作为预测的下一个词。
